# **1. Installation des librairies**

In [ ]:
!pip install captum
!pip install pytorch_lightning

# **2. Téléchargement du jeu de données de test**

In [ ]:
!wget https://nextcloud.ig.umons.ac.be/s/XqEMQtqQNPoG2cY/download/test.zip
!unzip test.zip -d /test
!rm test.zip

# **3. Importation des librarires**

In [ ]:
import torch
from torch.utils.data import DataLoader
import torchvision.datasets as datasets
import torchvision.transforms.v2 as transforms
import torch.nn as nn
from captum.attr import LayerGradCam
import matplotlib.pyplot as plt
from torchvision import models
import pytorch_lightning as pl
import numpy as np
import cv2

# **4. Chargement des données : test_loader**

In [ ]:
img_size = 224
testset_path  = "/test/test/"
batch_size = 1
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

test_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor()
    ])

test_dataset = datasets.ImageFolder(testset_path, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

# **5. Chargement du modèle**

In [ ]:
class FireDetectionModel(pl.LightningModule):
    def __init__(self, num_classes=3, learning_rate=3e-4):
        super().__init__()
        self.model = models.efficientnet_v2_s(weights="DEFAULT")
        self.model.classifier[1] = nn.Linear(self.model.classifier[1].in_features, num_classes)
        self.criterion = nn.CrossEntropyLoss()
        self.learning_rate = learning_rate

    def forward(self, x):
        return self.model(x)

model = FireDetectionModel.load_from_checkpoint("testmodel-sidi.ckpt")
model.eval()


# **6. Application de la méthode XAI "GradCAM" sur la première image**

In [ ]:
def get_last_conv_layer(model):
    last_conv_layer = None
    for module in model.modules():
        if isinstance(module, nn.Conv2d):
            last_conv_layer = module
    if last_conv_layer is None:
        raise ValueError("No Conv2d layer found in the model.")
    return last_conv_layer

def visualize_gradcam(model, inputs, labels, device, figsize=(18, 6)):
    # Get last convolutional layer
    last_conv_layer = get_last_conv_layer(model)
    gradcam = LayerGradCam(model, last_conv_layer)

    # Convert labels if necessary
    if isinstance(labels, torch.Tensor):
        labels = int(torch.argmax(labels.detach()).cpu().numpy())

    # Compute Grad-CAM attribution
    attribution = gradcam.attribute(inputs, target=labels, relu_attributions=True)

    # Convert tensors to numpy arrays
    attribution_np = attribution[0].cpu().detach().numpy()
    inputs_np = inputs[0].cpu().permute(1, 2, 0).detach().numpy()

    # Get original image dimensions
    height, width = inputs_np.shape[:2]

    # Resize attribution map to match input image size
    heatmap = cv2.resize(attribution_np[0], (width, height))

    # Normalize heatmap
    heatmap = np.maximum(heatmap, 0)
    heatmap = heatmap / np.max(heatmap)

    # Plot results
    fig, axes = plt.subplots(1, 3, figsize=figsize)

    # Original image
    axes[0].imshow(inputs_np)
    axes[0].set_title('Original Image')
    axes[0].axis('off')

    # Heatmap only
    im = axes[1].imshow(heatmap, cmap='jet')
    axes[1].set_title('Attention Heatmap')
    axes[1].axis('off')
    plt.colorbar(im, ax=axes[1])

    # Overlay
    axes[2].imshow(inputs_np)
    axes[2].imshow(heatmap, alpha=0.5, cmap='jet')
    axes[2].set_title('Overlay')
    axes[2].axis('off')

    plt.tight_layout()
    return fig

inputs, labels = next(iter(test_loader))
inputs = inputs.to(device)

fig = visualize_gradcam(model, inputs, labels, device)
plt.show()